In [ ]:
#parse pdf -> chunk text -> create embedding -> store in faiss -> retireve top similar chunk

In [ ]:
!pip install faiss-cpu

In [ ]:
!pip install PyPDF2

In [ ]:
#load pdf
from PyPDF2 import PdfReader

def load_pdf(file_path):
  read = PdfReader(file_path)
  text = ''
  for page in read.pages:
    text += page.extract_text()
  return text.strip()

In [ ]:
#chunking
def chunk_text(text, chunk_size=150):
  chunks=[]
  for i in range(0, len(text), chunk_size):
    chunks.append(text[i:i+chunk_size])
  return chunks

In [ ]:
#embeddings
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/e5-small-v2")

def embed_chunk(chunks):
  chunks = ["passage: " + c for c in chunks]
  return model.encode(chunks)

In [ ]:
#faiss
import faiss
import numpy as np

def faiss_index(embeddings):
  embeddings = np.array(embeddings).astype("float32")
  faiss.normalize_L2(embeddings)

  dim = embeddings.shape[1]
  index = faiss.IndexFlatIP(dim) #inner product for cosine
  index.add(np.array(embeddings))
  return index

In [ ]:
def retrieve(query, model, index, chunks, top_k=3):
  query_embeddings = model.encode(['query: ' + query])
  query_embeddings = np.array(query_embeddings).astype("float32")

  faiss.normalize_L2(query_embeddings)
  #serach for top k nearest vectors to the query embedding
  distance, indices = index.search(query_embeddings, top_k) # distance -> score , lower -> more similar
  results = []

  for i, idx in enumerate(indices[0]):
    results.append({"chunk": chunks[idx], "score": float(distance[0][i])})

  return results

In [ ]:
#pipeline
def rag_pipeline(file_path, query):
  if not query:
    return{"error": "Query is empty"}

  text = load_pdf(file_path)
  if not text:
    return{"error": "empty pdf "}

  chunks = chunk_text(text)
  if not chunks:
    return{"error": "No chunks created"}

  embeddings = embed_chunk(chunks)
  index = faiss_index(embeddings)
  result = retrieve(query, model, index, chunks)

  return result

In [ ]:
result = rag_pipeline("/content/AML_projects(book).pdf", "color of sky")
print(result)

In [ ]:
text = load_pdf("/content/AML_projects(book).pdf")
chunks = chunk_text(text)
for i, c in enumerate(chunks):
  print(i, c[:100])